# Capítulo 6 · Laboratório 6: Fibonacci no preço e no tempo
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap06.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. Dos coelhos de Fibonacci às razões, dos pivôs aos clusters de Carolyn Boroden e aos níveis do Fibonaccing de Marco Rossi. Depois, os testes: as correções param mesmo em 38,2%, 50% e 61,8%? Os rompimentos falham 70% das vezes? As razões de Fibonacci no tempo acertam mais do que razões inventadas?

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap06.pkl"
pc = lambda x, k=1: f"{x:.{k}%}".replace(".", ",")
nr = lambda x, k=2: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
ACOES = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "B3SA3", "ABEV3", "WEGE3", "RENT3", "SUZB3", "GGBR4", "ITSA4",
         "AXIA3", "PRIO3", "RADL3", "LREN3", "BPAC11", "SBSP3", "EQTL3", "VIVT3", "CSNA3", "CSAN3", "RAIL3", "MGLU3",
         "USIM5", "KLBN11", "TOTS3", "HYPE3", "ENEV3", "BRAP4", "CYRE3", "EMBJ3", "UGPA3", "CMIG4", "GOAU4"]
EXTRAS = {"^BVSP": "2000-01-01", "^GSPC": "1928-01-01", "ETH-USD": "2017-11-01"}


def baixa(t, ini):
    d = yf.download(t, start=ini, end=FIM, auto_adjust=False, progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    return d.dropna(subset=["Close"])


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


D = pd.read_pickle(CACHE) if os.path.exists(CACHE) else {}
for t in [a + ".SA" for a in ACOES] + list(EXTRAS):
    if t not in D:
        D[t] = baixa(t, EXTRAS.get(t, "2015-01-01"))
confere({t: D[t] for t in EXTRAS})
pd.to_pickle(D, CACHE)

## Parte 1: dados

In [ ]:
def limpa(d, indice=False):
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] >= d["Low"])
    if not indice:
        ok &= d["Volume"] > 0
    return d[ok].copy()


B = {a: limpa(D[a + ".SA"]) for a in ACOES}
IBOV = limpa(D["^BVSP"], True)
SPX = limpa(D["^GSPC"], True)
ETH = limpa(D["ETH-USD"], True)
print(f"Parte 1. {len(ACOES)} ações (2015 a 2026), Ibovespa (desde 2000), S&P 500 (desde 1928) e Ethereum (desde 2017).")

## Parte 2: os coelhos e as razões

In [ ]:
def coelhos(meses=12):
    """Pares adultos e jovens mês a mês: cada par adulto gera um par novo por mês; o jovem vira adulto em um mês."""
    adultos, jovens, linhas = 1, 0, []
    for m in range(1, meses + 1):
        adultos, jovens = adultos + jovens, adultos
        linhas.append((m, adultos, jovens, adultos + jovens))
    return pd.DataFrame(linhas, columns=["mês", "adultos", "recém-nascidos", "total de pares"])


C = coelhos()
fib = [1, 1]
while len(fib) < 25:
    fib.append(fib[-1] + fib[-2])
print("\nParte 2. O problema dos coelhos (pares ao fim de cada mês):", list(C["total de pares"]))
print(f"   Razão entre vizinhos: 144/89 = {144 / 89:.6f}; 233/144 = {233 / 144:.6f}; 46368/28657 = {fib[23] / fib[22]:.9f}"
      f" (phi = {(1 + 5 ** 0.5) / 2:.9f})")
RAZOES = {"0,236": 0.618 - 0.382, "0,382": 1 - 0.618, "0,5": 0.5, "0,618": 0.618, "0,786": 0.618 ** 0.5,
          "1,272": 1.618 ** 0.5, "1,618": 1.618, "2,618": 1.618 ** 2, "4,236": 1.618 ** 3}
print("   Razões derivadas:", {k: round(v, 3) for k, v in RAZOES.items()})

## Parte 3: pivôs pelo ziguezague

In [ ]:
def zigzag(d, limite=0.05):
    """Topos e fundos pela máxima e pela mínima: [(posição, preço, 'T' ou 'F')]."""
    h, l = d["High"].to_numpy(), d["Low"].to_numpy()
    piv, tend, imax, imin = [], 0, 0, 0
    for i in range(1, len(h)):
        if h[i] > h[imax]:
            imax = i
        if l[i] < l[imin]:
            imin = i
        if tend >= 0 and l[i] < h[imax] * (1 - limite) and (not piv or piv[-1][2] == "F"):
            piv.append((imax, h[imax], "T"))
            tend = -1
            imin = imax + 1 + int(np.argmin(l[imax + 1:i + 1])) if imax < i else i
        elif tend <= 0 and h[i] > l[imin] * (1 + limite) and (not piv or piv[-1][2] == "T"):
            piv.append((imin, l[imin], "F"))
            tend = 1
            imax = imin + 1 + int(np.argmax(h[imin + 1:i + 1])) if imin < i else i
    return piv


print("\nParte 3. Pivôs do Ibovespa (ziguezague de 6%) no último ano:")
for i, p, t in [x for x in zigzag(IBOV, 0.06) if IBOV.index[x[0]] >= pd.Timestamp("2025-06-01")]:
    print(f"   {t} {IBOV.index[i]:%d/%m/%Y} {nr(p, 0)}")

## Parte 4: retrações, extensões e projeções

In [ ]:
def retracoes(a, b, razoes=(0.236, 0.382, 0.5, 0.618, 0.786)):
    """De a (início da perna) até b (fim): níveis de correção a partir de b."""
    return {r: b - r * (b - a) for r in razoes}


def extensoes(a, b, razoes=(1.272, 1.618, 2.618)):
    """Perna de a até b; a extensão passa do ponto a (o início da perna), do outro lado."""
    return {r: b - r * (b - a) for r in razoes}


def projecoes(a, b, c, razoes=(0.618, 1.0, 1.618)):
    """Perna de a até b repetida a partir de c (mesma direção)."""
    return {r: c + r * (b - a) for r in razoes}


ib = IBOV.loc["2025-06-01":]
zz = [x for x in zigzag(IBOV, 0.06) if IBOV.index[x[0]] >= ib.index[0]]
F0 = min((x for x in zz if x[2] == "F"), key=lambda x: x[1])
T0 = max((x for x in zz if x[2] == "T"), key=lambda x: x[1])
print("\nParte 4. Exemplo no Ibovespa: retrações da alta de"
      f" {nr(F0[1], 0)} ({IBOV.index[F0[0]]:%d/%m/%Y}) até {nr(T0[1], 0)} ({IBOV.index[T0[0]]:%d/%m/%Y})")
for r, v in retracoes(F0[1], T0[1]).items():
    print(f"   retração de {pc(r)}: {nr(v, 0)}")

## Parte 5: clusters de Carolyn Boroden

In [ ]:
def clusters(d, limite=0.08, faixa=0.012, minimo=3, pernas=6):
    """Retrações, extensões e projeções das últimas pernas; zonas onde 3 ou mais níveis caem perto."""
    pv = zigzag(d, limite)[-(pernas + 1):]
    ultimo = d["Close"].iloc[-1]
    niveis = []
    for (i0, a, _), (i1, b, _) in zip(pv, pv[1:]):
        niveis += [("ret", v) for v in retracoes(a, b, (0.382, 0.5, 0.618, 0.786)).values()]
        niveis += [("ext", v) for v in extensoes(a, b, (1.272, 1.618)).values()]
    for (_, a, _), (_, b, _), (_, c, _) in zip(pv, pv[1:], pv[2:]):
        niveis += [("proj", v) for v in projecoes(a, b, c, (1.0, 1.618)).values()]
    niveis = sorted(v for _, v in niveis if 0.7 * ultimo < v < 1.3 * ultimo)
    grupos, atual = [], [niveis[0]]
    for v in niveis[1:]:
        if v <= atual[0] * (1 + faixa):
            atual.append(v)
        else:
            grupos.append(atual)
            atual = [v]
    grupos.append(atual)
    return ultimo, [(min(g), max(g), len(g)) for g in grupos if len(g) >= minimo], pv


print("\nParte 5. Clusters (3 ou mais níveis a menos de 1,2% um do outro) em 30/09/2026:")
CL = {}
for a in ("BPAC11", "ITUB4", "SBSP3"):
    ultimo, zonas, pv = clusters(B[a])
    CL[a] = (ultimo, zonas, pv)
    print(f"   {a} (último R$ {nr(ultimo)}): " + "; ".join(f"R$ {nr(x)} a {nr(y)} ({n} níveis)" for x, y, n in zonas))

In [ ]:
import matplotlib.pyplot as plt

def mostra_clusters(ativo):
    """Fechamento dos últimos pregões com as zonas de cluster em cinza."""
    ultimo, zonas, pv = CL[ativo]
    d = B[ativo].iloc[pv[0][0] - 10:]
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(d.index, d["Close"], color="black", lw=0.9)
    for lo, hi, n in zonas:
        ax.axhspan(lo, hi, color="gray", alpha=0.15 * n)
    ax.set_title(f"{ativo}: zonas com 3 ou mais níveis de Fibonacci (mais escuro = mais níveis)")
    plt.show()

for a in ("BPAC11", "ITUB4", "SBSP3"):
    mostra_clusters(a)

## Parte 6: as correções param nos níveis de Fibonacci?

In [ ]:
PROF = []
for a in ACOES:
    pv = zigzag(B[a], 0.10)
    for (_, a1, _), (_, b1, _), (_, c1, _) in zip(pv, pv[1:], pv[2:]):
        PROF.append(abs(c1 - b1) / abs(b1 - a1))
PROF = np.array([p for p in PROF if 0.2 <= p <= 0.9])
perto = lambda nivel, tol=0.02: float(np.mean(np.abs(PROF - nivel) <= tol))
GRADE = pd.Series({round(x, 3): perto(x) for x in np.arange(0.25, 0.851, 0.01)})
print(f"\nParte 6. Correções de pernas de 10% ou mais nas 35 ações, entre 20% e 90% de profundidade: {len(PROF)}")
for nivel in (0.382, 0.5, 0.618, 0.45, 0.55, 0.70):
    pos = int((GRADE > perto(nivel)).sum()) + 1
    print(f"   a ±2 p.p. de {pc(nivel)}: {pc(perto(nivel))} das correções; posição {pos} entre os {len(GRADE)} níveis"
          f" de 25% a 85% (1 = o mais visitado)")
print(f"   média de um nível qualquer entre 25% e 85%: {pc(GRADE.mean())}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.hist(PROF, bins=np.arange(0.2, 0.901, 0.01), color="lightgray", edgecolor="gray")
for r in (0.382, 0.5, 0.618):
    ax.axvline(r, color="black", ls="--")
ax.set_title("Profundidade das correções (as linhas são 38,2%, 50% e 61,8%)")
plt.show()

## Parte 7: "70% dos rompimentos falham"?

In [ ]:
TOTAL = FALHAS = 0
for a in ACOES:
    d = B[a]
    c, h = d["Close"].to_numpy(), d["High"].to_numpy()
    nivel = pd.Series(h).rolling(20).max().shift().to_numpy()
    for i in range(21, len(d) - 5):
        if c[i] > nivel[i] and c[i - 1] <= nivel[i - 1]:
            TOTAL += 1
            FALHAS += bool((c[i + 1:i + 6] < nivel[i]).any())
print(f"\nParte 7. Rompimentos da máxima de 20 pregões: {TOTAL}; voltaram a fechar abaixo do nível em até 5 pregões:"
      f" {pc(FALHAS / TOTAL, 0)}")

## Parte 8: os níveis do Fibonaccing no Ibovespa

In [ ]:
NIVEIS_ROSSI = [-0.118, 0, 0.236, 0.764, 1.0, 1.118, 1.272, 1.414, 1.618]


def fibonaccing(d, fundo, topo, depois):
    """0% no fundo e 100% no topo da perna; depois, a ordem em que o preço tocou cada nível."""
    i0, i1 = fundo, topo
    a, b = d["Low"].iloc[i0], d["High"].iloc[i1]
    nv = {r: a + r * (b - a) for r in NIVEIS_ROSSI}
    seq, ult = [], None
    for k in range(i1 + 1, min(i1 + 1 + depois, len(d))):
        lo, hi = d["Low"].iloc[k], d["High"].iloc[k]
        tocados = [r for r, v in nv.items() if lo <= v <= hi]
        for r in tocados:
            if r != ult:
                seq.append((d.index[k], r))
                ult = r
    return nv, seq


ibz = [x for x in zigzag(IBOV, 0.06) if IBOV.index[x[0]] >= pd.Timestamp("2025-06-01")]
FUNDO = min((x for x in ibz if x[2] == "F"), key=lambda x: x[1])
TOPO = max((x for x in ibz if x[2] == "T" and x[0] > FUNDO[0]), key=lambda x: x[1])
NV, SEQ = fibonaccing(IBOV, FUNDO[0], TOPO[0], 120)
print(f"\nParte 8. Fibonaccing no Ibovespa: 0% em {nr(FUNDO[1], 0)} ({IBOV.index[FUNDO[0]]:%d/%m/%Y}), 100% em"
      f" {nr(TOPO[1], 0)} ({IBOV.index[TOPO[0]]:%d/%m/%Y})")
print("   níveis:", {f"{r:.1%}": nr(v, 0) for r, v in NV.items()})
print("   ordem dos toques depois do topo:", " -> ".join(f"{r:.1%} ({dt:%d/%m})" for dt, r in SEQ[:12]))

## Parte 9: Fibonacci no tempo

In [ ]:
SERIES = {"IBOV": IBOV.loc["1995":], "SPX": SPX.loc["1950":]}
for a in ("PETR4", "VALE3", "ITUB4", "BBDC4", "ABEV3", "WEGE3", "BBAS3", "GGBR4"):
    SERIES[a] = B[a]
N_AB, N_BC = [], []
for s in SERIES.values():
    pv = zigzag(s, 0.10)
    for (i0, _, _), (i1, _, _), (i2, _, _) in zip(pv, pv[1:], pv[2:]):
        if i1 > i0 and i2 > i1:
            N_AB.append(i1 - i0)
            N_BC.append(i2 - i1)
N_AB, N_BC = np.array(N_AB), np.array(N_BC)
LONGA = N_AB >= 20          # pernas curtas geram razões "redondas" por acaso (5/5, 10/5...)
RT = N_BC[LONGA] / N_AB[LONGA]
FIBT = np.array([0.382, 0.5, 0.618, 0.786, 1.0, 1.618, 2.0, 2.618])
INVENT = np.array([0.45, 0.7, 0.9, 1.15, 1.3, 1.8, 2.3, 3.0])
pertot = lambda r, alvos, tol=0.05: float((np.abs(r[:, None] / alvos[None, :] - 1) <= tol).any(axis=1).mean())
FIB_V = pertot(RT, FIBT)
VIZ_V = {k: pertot(RT, FIBT * k) for k in (0.9, 1.1)}
SIM = np.corrcoef(np.log(N_AB), np.log(N_BC))[0, 1]
print(f"\nParte 9. Duração da correção / duração da perna anterior ({len(RT)} pares com perna de 20 pregões ou mais;"
      f" Ibovespa, S&P 500 e 8 ações):")
print(f"   perto (±5%) de uma das 8 razões de Fibonacci: {pc(FIB_V)}; das mesmas razões 10% abaixo: {pc(VIZ_V[0.9])};"
      f" 10% acima: {pc(VIZ_V[1.1])}; de 8 razões inventadas: {pc(pertot(RT, INVENT))}")
print(f"   correlação entre a duração de uma perna e a da seguinte: {SIM:.2f}".replace(".", ","))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.hist(RT, bins=np.exp(np.arange(np.log(0.2), np.log(4.0), 0.04)), color="lightgray", edgecolor="gray")
ax.set_xscale("log")
for r in FIBT:
    ax.axvline(r, color="black", ls="--", lw=0.8)
ax.set_title("Duração da correção ÷ duração da perna anterior (linhas: razões de Fibonacci)")
plt.show()

## Parte 10: janelas de tempo

In [ ]:
def janelas(d, limite=0.08, tol=2, razoes=(0.618, 1.0, 1.618)):
    """Para cada trio de pivôs A, B, C: projeta a duração de A até B a partir de C. Acertou se um pivô novo
    aparece a até 'tol' pregões de alguma janela antes de passar da última."""
    pv = zigzag(d, limite)
    out = []
    for k in range(len(pv) - 3):
        (ia, _, _), (ib_, _, _), (ic, _, _) = pv[k:k + 3]
        dur = ib_ - ia
        alvos = [ic + round(r * dur) for r in razoes]
        prox = pv[k + 3][0]
        acerto = any(abs(prox - x) <= tol for x in alvos)
        out.append(dict(a=ia, b=ib_, c=ic, alvos=alvos, prox=prox, acerto=acerto, data=d.index[ic]))
    return pd.DataFrame(out)


JP, JV = janelas(B["PETR4"]), janelas(B["VALE3"])
TODAS = pd.concat([janelas(B[a]) for a in ACOES])
print(f"\nParte 10. Janelas de tempo (61,8%, 100% e 161,8% da perna A-B, a partir de C; tolerância de 2 pregões):"
      f" acertaram {pc(TODAS['acerto'].mean())} das {len(TODAS)} projeções nas 35 ações")
ALT = {rz: pd.concat([janelas(B[a], razoes=rz) for a in ACOES])["acerto"].mean()
       for rz in ((0.55, 1.0, 1.8), (0.7, 1.4, 2.3))}
for rz, v in ALT.items():
    print(f"   com razões inventadas {rz}: {pc(v)}".replace(".", ","))
J_OK = JP[JP["acerto"] & (JP["prox"] - JP["c"] >= 30) & (JP["data"] >= "2024-06-01")].iloc[-1]   # pivô longe de C
J_ERR = JV[~JV["acerto"] & (JV["data"] >= "2024-06-01")].iloc[-1]
print(f"   PETR4 (acertou): C em {B['PETR4'].index[J_OK['c']]:%d/%m/%Y}; pivô seguinte em"
      f" {B['PETR4'].index[J_OK['prox']]:%d/%m/%Y}")
print(f"   VALE3 (errou): C em {B['VALE3'].index[J_ERR['c']]:%d/%m/%Y}; pivô seguinte em"
      f" {B['VALE3'].index[J_ERR['prox']]:%d/%m/%Y}")

## Parte 11: o teste do tempo de Prechter

In [ ]:
s = SPX["Close"]
TOPO_SPX = s.loc["2021":"2022"].idxmax()
FUNDO_SPX = s.loc[TOPO_SPX:"2022"].idxmin()
VOLTA = s.loc["2023":][s.loc["2023":] > s[TOPO_SPX]].index[0]
print(f"\nParte 11. Prechter (janeiro de 2021, topo em 2021 por anos de Fibonacci): máxima de fechamento do S&P 500 em"
      f" {TOPO_SPX:%d/%m/%Y} ({nr(s[TOPO_SPX])}); queda de {pc(1 - s[FUNDO_SPX] / s[TOPO_SPX], 0)} até"
      f" {FUNDO_SPX:%d/%m/%Y}; nova máxima em {VOLTA:%d/%m/%Y}")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a função `clusters` da Parte 5. Mostre de onde saem os níveis (retrações, extensões e projeções das últimas seis pernas do ziguezague de 8%), como a faixa de 1,2% junta os níveis vizinhos e onde está o mínimo de 3 níveis da definição de Carolyn Boroden." Confira com a Parte 5: no BPAC11, as duas zonas de 4 níveis vão de R$ 57,11 a R$ 57,69 e de R$ 53,32 a R$ 53,91, como nos Gráficos 6.7 a 6.9.
2. **Adaptar:** "Crie uma célula nova no fim do notebook que baixe [ATIVO], com o ticker do Yahoo Finance, pelas funções `baixa` e `limpa`, e rode `clusters` nele. Não acrescente o ativo à lista `ACOES`, porque isso muda os testes das Partes 6, 7 e 10, e não mude as razões, a faixa de 1,2% nem o ziguezague de 8%." Antes de olhar o ativo novo, confira se a Parte 5 continua mostrando as zonas do BPAC11 do livro.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 6, troque a tolerância de 2 para 1 ponto percentual. Algum nível de Fibonacci sobe no ranking?
2. Na Parte 5, ponha as suas ações em clusters() e compare as zonas com o gráfico do seu home broker.
3. Na Parte 7, troque a máxima de 20 pregões pela de 55. Os rompimentos falham menos?
4. Na Parte 10, crie as suas razões inventadas e compare com 61,8%, 100% e 161,8%.

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.